# Querying alert brokers

## Lasair (LSST)

[Lasair](https://lasair.lsst.ac.uk) is the UK alert broker for Rubin/LSST. This notebook sets up
API access and pulls every object with a detection in the past *N* days.

**Token:** log in at <https://lasair.lsst.ac.uk>, click your name (top right) -> *My Profile*,
copy the API token, then either

```bash
echo 'YOUR_TOKEN_HERE' > ~/.lasair_token && chmod 600 ~/.lasair_token
```

or set `LASAIR_TOKEN` in the environment. The auth cell picks up either.

**Rate limits:** 10 calls/hour anonymous, 100/hour with an account, 10000/hour for "power" users.
Every page of a paged query is one call — a 7-day pull is ~25 calls at the maximum page size — so
`count_recent()` is worth running before `recent_objects()`.

In [8]:
# pip install lasair
import os, time
from pathlib import Path

import pandas as pd
import lasair
from astropy.time import Time

### Authentication

In [9]:
def get_token():
    """Token from $LASAIR_TOKEN, else ~/.lasair_token."""
    token = os.environ.get("LASAIR_TOKEN")
    if token:
        return token.strip()
    path = Path.home() / ".lasair_token"
    if path.exists():
        return path.read_text().strip()
    raise RuntimeError(
        "No Lasair token found. Put it in ~/.lasair_token or set $LASAIR_TOKEN "
        "(https://lasair.lsst.ac.uk -> your name -> My Profile)."
    )


# Responses are cached on disk, so re-running a cell doesn't spend quota.
CACHE = Path("lasair_cache")
CACHE.mkdir(exist_ok=True)

# Default endpoint is the LSST instance, https://api.lasair.lsst.ac.uk/api .
# For ZTF, pass endpoint="https://api.lasair-ztf.lsst.ac.uk/api".
L = lasair.lasair_client(get_token(), cache=str(CACHE), timeout=300)

L.query(selected="diaObjectId", tables="objects", conditions="", limit=1)  # smoke test

[{'diaObjectId': 170657504701711053}]

### What "alerts from the past N days" means here

The API's `query` endpoint serves the **object** table — one row per `diaObjectId` with summary
photometry and first/last detection times — not individual alert packets. So the query below
returns every object whose *most recent* detection (`lastDiaSourceMjdTai`) lands in the window.
Per-alert detections for a single object come from `L.object()` (bottom of the notebook).

Times are MJD in the **TAI** scale, hence `Time.now().tai.mjd` rather than `.mjd`.

Note that Lasair's ingest runs behind the telescope — the cell below reports by how much. If a
short window comes back empty, that lag is usually why.

In [ ]:
def latest_detection():
    """(mjd, days_behind) of the most recent detection anywhere in the objects table."""
    row = L.query(selected="max(lastDiaSourceMjdTai) AS mx", tables="objects",
                  conditions="", limit=1)[0]
    mjd = row["mx"]
    return mjd, Time.now().tai.mjd - mjd


mjd, lag = latest_detection()
print(f"latest detection: MJD {mjd:.4f} = {Time(mjd, format='mjd', scale='tai').iso} "
      f"({lag:.2f} days ago)")

### The query

In [14]:
# Verified column names; see https://lasair.lsst.ac.uk/schema/ for the full list,
# or run L.query(selected="*", tables="objects", conditions="", limit=1).
OBJECT_COLUMNS = [
    "objects.diaObjectId",
    "objects.ra",
    "objects.decl",
    "objects.firstDiaSourceMjdTai",
    "objects.lastDiaSourceMjdTai",
    "objects.nDiaSources",
    "objects.latest_psfFlux",
    "objects.absMag",  # only set where Sherlock has a distance: ~98% null
]
SHERLOCK_COLUMNS = ["sherlock_classifications.classification"]

MAX_PAGE = 10_000  # server-side cap on rows per call


def _window(n_days, with_sherlock, extra_conditions):
    """(tables, conditions) for the past-n_days cut."""
    mjd_now = Time.now().tai.mjd
    tables = "objects"
    conditions = f"objects.lastDiaSourceMjdTai > {mjd_now - n_days:.6f}"
    if with_sherlock:
        tables += ", sherlock_classifications"
        conditions += " AND objects.diaObjectId = sherlock_classifications.diaObjectId"
    if extra_conditions:
        conditions += f" AND ({extra_conditions})"
    return mjd_now, tables, conditions


def count_recent(n_days, with_sherlock=True, extra_conditions=None):
    """How many objects a recent_objects() call would return. Costs one API call."""
    _, tables, conditions = _window(n_days, with_sherlock, extra_conditions)
    return L.query(selected="count(*) AS n", tables=tables, conditions=conditions, limit=1)[0]["n"]


def recent_objects(n_days, columns=None, with_sherlock=True, extra_conditions=None,
                   page_size=MAX_PAGE, max_calls=50, verbose=True):
    """All Lasair objects with a detection in the past `n_days`, as a DataFrame.

    with_sherlock   : join the Sherlock context classification. It is an INNER join, so
                      objects Sherlock has not classified are dropped (~40% of a week's
                      objects). Pass False for the unfiltered object list.
    extra_conditions: extra SQL ANDed onto the time cut, e.g.
                      "sherlock_classifications.classification = \'SN\'".
    max_calls       : safety stop, since each page is one call against the hourly quota.

    Pages by keyset on diaObjectId rather than OFFSET, so rows cannot be skipped or
    duplicated if the table changes mid-pull.
    """
    if columns is None:
        columns = OBJECT_COLUMNS + (SHERLOCK_COLUMNS if with_sherlock else [])
    page_size = min(page_size, MAX_PAGE)
    mjd_now, tables, conditions = _window(n_days, with_sherlock, extra_conditions)
    selected = ", ".join(columns)

    rows, last_id = [], None
    for call in range(max_calls):
        page_conditions = conditions
        if last_id is not None:
            page_conditions += f" AND objects.diaObjectId > {last_id}"
        page_conditions += " ORDER BY objects.diaObjectId"

        page = L.query(selected=selected, tables=tables, conditions=page_conditions,
                       limit=page_size)
        if isinstance(page, dict) and "error" in page:
            raise RuntimeError(page["error"])
        rows.extend(page)
        if verbose:
            print(f"call {call + 1}: +{len(page)} rows (total {len(rows)})")
        if len(page) < page_size:
            break
        last_id = page[-1]["diaObjectId"]
    else:
        print(f"Warning: stopped at max_calls={max_calls}; results are truncated.")

    df = pd.DataFrame(rows)
    if df.empty:
        _, lag = latest_detection()
        print(f"No objects in the last {n_days} days — the most recent detection in Lasair "
              f"is {lag:.2f} days old.")
        return df

    df["days_ago"] = (mjd_now - df["lastDiaSourceMjdTai"]).round(2)
    return df.sort_values("days_ago").reset_index(drop=True)

Size the query first — 10000 rows per call is the server cap, so this is `count / 10000` calls:

In [ ]:
N_DAYS = 5
n = count_recent(N_DAYS)
print(f"{n} objects in the last {N_DAYS} days -> {-(-n // MAX_PAGE)} API calls")

In [ ]:
df = recent_objects(N_DAYS)
print(len(df), "objects")
df.head(10)

In [ ]:
df["classification"].value_counts()

In [ ]:
df.to_csv(f"lasair_recent_{N_DAYS}d.csv", index=False)

### Narrower cuts

`extra_conditions` takes any SQL that the joined tables support, which is usually much cheaper
than pulling everything and filtering in pandas.

In [ ]:
sne = recent_objects(
    n_days=7,
    extra_conditions="sherlock_classifications.classification = 'SN' AND objects.nDiaSources > 5",
)
print(len(sne), "SN candidates")
sne.head()

### Individual alerts for one object

`L.object()` returns the machine-readable object page: `diaSourcesList` holds the detections
(the alerts themselves) and `diaForcedSourcesList` the forced photometry. `lite=False` returns
every column of each source rather than the summary set.

`lasair_added=False` skips Lasair's derived extras (image URLs and such). That is the default in
the wrapper below because the endpoint intermittently answers
`Bad Request: {"error": "'image_urls'"}` for some objects — a server-side bug, not a bad request —
and the sources themselves are unaffected.

In [ ]:
def get_object(diaObjectId, lite=True, lasair_added=False):
    """Object page including its individual sources."""
    return L.object(int(diaObjectId), lite=lite, lasair_added=lasair_added)


obj = get_object(df.loc[0, "diaObjectId"])
print(list(obj.keys()))

sources = pd.DataFrame(obj["diaSourcesList"])
print(len(sources), "detections,", len(obj["diaForcedSourcesList"]), "forced photometry points")
sources.head()

In [ ]:
# Sky context: what Sherlock thinks this object is associated with.
L.sherlock_object(int(df.loc[0, "diaObjectId"]), lite=False)["classifications"]

### Other calls

```python
L.cone(ra=194.494, dec=48.851, radius=240.0, requestType="all")  # radius in arcsec, max 1000
L.sherlock_position(ra, dec, lite=False)                          # context for any position
```

Tables available to `query()`: `objects`, `objects_ext`, `sherlock_classifications`,
`crossmatch_tns`, `watchlist_hits`, and annotator tables. Columns:
<https://lasair.lsst.ac.uk/schema/>.

For a push feed instead of polling, Lasair publishes Kafka streams of filter output via
`lasair.lasair_consumer` — see <https://lasair.readthedocs.io/>.

## Looking up an external id list

Cross-matching an ALeRCE export against Lasair: take the `oid` column of
`alerce/alerce_lsstDiaObj_top200k_19Nov26.fits` (200,001 LSST `diaObjectId`s) and pull the Lasair
row for each, with Sherlock's host association in the same call.

**Cost.** Lasair rejects a `conditions` string longer than **4096 characters**, which fits **214**
of these 18-digit ids per `IN (...)` clause — so the whole file is ~935 calls against an account
quota of 100 calls/hour, roughly 10 hours. A power-user token (10,000/hour) turns that into about
20 minutes. Responses are cached in `lasair_cache/`, and the loop checkpoints to disk and resumes,
so it is safe to interrupt.

**One call per batch means an inner join.** Naming `sherlock_classifications` in `tables` makes the
server add `objects.diaObjectId = sherlock_classifications.diaObjectId` to the WHERE clause itself,
so ids Sherlock has not classified do not come back at all. There is no way around that in a single
query: `LEFT JOIN` in the `tables` field is silently stripped and produces malformed SQL, and a
correlated subquery in `selected` is refused outright (*"Cannot use the word SELECT in the SELECT
clause"*). So pick what you need:

| `mode` | calls per batch | what you get |
|---|---|---|
| `"hosts"` | 1 | object + Sherlock host columns, for the ~24% of these ids Sherlock has classified |
| `"objects"` | 1 | object columns only, for every id (100% of this file matched in testing) |

Run it twice with the two modes if you want both — that is the two-pass version, at double the quota.


In [10]:
import json
from pathlib import Path

import numpy as np
from astropy.io import fits

ID_SOURCE = Path("alerce/alerce_lsstDiaObj_top100k_ndiasourcesGrt10_19Nov26.fits")
CONDITIONS_MAX = 4096   # Lasair's limit on the `conditions` field


def read_oids(path=ID_SOURCE, column="oid"):
    """Sorted, unique diaObjectIds from a FITS table."""
    with fits.open(path) as hdul:
        return np.unique(hdul[1].data[column].astype("int64"))


oids = read_oids()
print(f"{len(oids)} unique ids, {oids.min()} .. {oids.max()}")

100000 unique ids, 170019695480537105 .. 314051321274564691


In [11]:
OBJECT_FIELDS = ("objects.diaObjectId, objects.ra, objects.decl, objects.nDiaSources, "
                 "objects.firstDiaSourceMjdTai, objects.lastDiaSourceMjdTai, objects.latest_psfFlux")
SHERLOCK_FIELDS = ("sherlock_classifications.classification, sherlock_classifications.raDeg, "
                   "sherlock_classifications.decDeg, sherlock_classifications.separationArcsec, "
                   "sherlock_classifications.physical_separation_kpc, sherlock_classifications.z, "
                   "sherlock_classifications.photoZ, sherlock_classifications.catalogue_table_name")

MODES = {
    # mode: (selected, tables) -- the server adds the join condition for the second table
    "hosts":   (OBJECT_FIELDS + ", " + SHERLOCK_FIELDS, "objects, sherlock_classifications"),
    "objects": (OBJECT_FIELDS, "objects"),
}

# Server-side cuts for mode="hosts". The redshift is that of Sherlock's matched source,
# which is the *host* only for transients offset from a galaxy (SN) or in its nucleus (NT).
# For VS/AGN it is the star/AGN itself, so those classes are excluded here.
HAS_HOST_Z = ("(sherlock_classifications.z IS NOT NULL "
              "OR sherlock_classifications.photoZ IS NOT NULL)")
HOST_Z_TRANSIENTS = HAS_HOST_Z + " AND sherlock_classifications.classification IN ('SN','NT')"


def lasair_for_ids(oids, mode="hosts", extra_conditions=None, out_csv=None, resume=True,
                   verbose=True):
    """Fetch Lasair rows for every id in `oids`, one query per batch.

    mode="hosts"     object + Sherlock host columns; inner join, so only classified ids return
    mode="objects"   object columns only, for every id
    extra_conditions SQL ANDed onto each batch, e.g. HOST_Z_TRANSIENTS (needs mode="hosts")

    Batches ids into `IN (...)` clauses as large as the 4096-character `conditions`
    limit allows (less whatever extra_conditions takes). With `out_csv`, each batch is
    appended as it arrives and the call count is checkpointed to a sidecar `.progress`
    file, so the loop is safe to interrupt and a re-run continues from there (delete
    both files to start over).
    """
    selected, tables = MODES[mode]
    prefix = "objects.diaObjectId IN ("
    suffix = ")" + (f" AND ({extra_conditions})" if extra_conditions else "")

    id_width = len(str(int(oids[0]))) + 1
    per_batch = max(1, (CONDITIONS_MAX - len(prefix) - len(suffix)) // id_width)
    batches = [oids[i:i + per_batch] for i in range(0, len(oids), per_batch)]

    out_csv = Path(out_csv) if out_csv else None
    progress = out_csv.with_suffix(".progress") if out_csv else None

    # The checkpoint records which query it belongs to, so changing the input, the
    # mode or the cuts starts a fresh run instead of silently skipping the wrong batches.
    stamp = {"n_ids": len(oids), "per_batch": per_batch, "mode": mode,
             "extra_conditions": extra_conditions}
    done = 0
    if resume and progress and progress.exists():
        saved = json.loads(progress.read_text())
        if all(saved.get(k) == v for k, v in stamp.items()):
            done = saved["done"]
            if verbose and done:
                print(f"resuming after {done} of {len(batches)} calls already in {out_csv}")
        else:
            if verbose:
                print(f"{progress} belongs to a different query; starting over")
            out_csv.unlink(missing_ok=True)
            progress.unlink()

    if verbose:
        print(f"{len(oids)} ids -> {per_batch} per call -> {len(batches)} calls "
              f"(~{len(batches) / 100:.1f} h at the 100 calls/hour account quota)")

    rows = []
    for i, batch in enumerate(batches):
        if i < done:
            continue
        conditions = prefix + ",".join(str(int(o)) for o in batch) + suffix
        assert len(conditions) <= CONDITIONS_MAX
        chunk = L.query(selected=selected, tables=tables, conditions=conditions, limit=10000)
        if isinstance(chunk, dict) and "error" in chunk:
            raise RuntimeError(chunk["error"])
        rows.extend(chunk)
        if out_csv:
            if chunk:
                pd.DataFrame(chunk).to_csv(out_csv, mode="a", index=False,
                                           header=not out_csv.exists())
            progress.write_text(json.dumps({**stamp, "done": i + 1}))
        if verbose and (i + 1) % 10 == 0:
            print(f"  call {i + 1}/{len(batches)}: {len(rows)} rows this run")

    df = pd.DataFrame(rows)
    if out_csv and out_csv.exists() and done:
        df = pd.read_csv(out_csv)          # include rows written by an earlier run
    if verbose and not df.empty:
        print(f"{len(df)} rows for {len(oids)} ids ({100 * len(df) / len(oids):.1f}% matched)")
    return df

In [6]:
# A first slice, so the cell returns in under a minute. Set N_IDS = len(oids) for the
# whole file -- ~935 calls, so plan for the quota before starting it.
N_IDS = 2140

matched = lasair_for_ids(oids[:N_IDS], mode="hosts", out_csv="lasair_for_alerce_top100k.csv")
if not matched.empty:
    print(matched.classification.value_counts().to_string())
    print("\nwith a redshift:", (matched.z.notna() | matched.photoZ.notna()).sum(), "of", len(matched))
matched.head()

2140 ids -> 214 per call -> 10 calls (~0.1 h at the 100 calls/hour account quota)
  call 10/10: 2140 rows this run
2140 rows for 2140 ids (100.0% matched)
VS         1675
SN          337
AGN          58
ORPHAN       29
UNCLEAR      26
NT           12
HPMS          3

with a redshift: 68 of 2140


,diaObjectId,ra,decl,nDiaSources,firstDiaSourceMjdTai,lastDiaSourceMjdTai,latest_psfFlux,classification,raDeg,decDeg,separationArcsec,physical_separation_kpc,z,photoZ,catalogue_table_name
0,170019695480537105,51.928339,-29.044808,18,61088.093945,61358.273587,-5632.93,VS,51.928334,-29.044807,0.323,NaN,NaN,NaN,DESI/GAIA
1,170019695496790075,53.024724,-28.329677,12,61088.093945,61358.267723,1352.51,SN,53.012670,-28.358580,110.866,10.361,0.004526,NaN,LASR
2,170019695754215502,54.729393,-28.216583,17,61088.095500,61358.273587,6391.94,AGN,54.729363,-28.216578,0.125,NaN,NaN,NaN,PS1/GAIA
3,170019695791439922,52.365057,-27.485678,18,61088.095500,61358.273587,3011.97,AGN,52.365064,-27.485691,0.180,NaN,NaN,NaN,GAIA/PS1
4,170019695987523894,53.502262,-29.687707,16,61088.093945,61356.284459,-3879.09,AGN,53.502255,-29.687705,0.184,1.527,1.224000,NaN,MILLIQUAS/GAIA


### Only transients with a host redshift

`HOST_Z_TRANSIENTS` keeps ids where Sherlock calls the object `SN` or `NT` **and** its matched
host has a spectroscopic (`z`) or photometric (`photoZ`) redshift. The cut runs on the server, so
the non-matching rows are never downloaded. The extra SQL takes up room in the 4096-character
limit, so each batch holds 206 ids instead of 214 (~4% more calls).
Use `HAS_HOST_Z` alone if you want every class.

`z_best` prefers the spec-z and falls back to the photo-z; `z_is_spec` records which one you got.

In [12]:
hosted = lasair_for_ids(oids[:N_IDS], mode="hosts", extra_conditions=HOST_Z_TRANSIENTS,
                        out_csv="lasair_for_alerce_top100k_hostz.csv")
if not hosted.empty:
    hosted["z_is_spec"] = hosted.z.notna()
    hosted["z_best"] = hosted.z.fillna(hosted.photoZ)
    print(hosted.classification.value_counts().to_string())
    print(f"\nspec-z: {hosted.z_is_spec.sum()}, photo-z only: {(~hosted.z_is_spec).sum()}")
hosted.head()

resuming after 11 of 11 calls already in lasair_for_alerce_top100k_hostz.csv
2140 ids -> 206 per call -> 11 calls (~0.1 h at the 100 calls/hour account quota)
54 rows for 2140 ids (2.5% matched)
SN    44
NT    10

spec-z: 25, photo-z only: 29


,diaObjectId,ra,decl,nDiaSources,firstDiaSourceMjdTai,lastDiaSourceMjdTai,latest_psfFlux,classification,raDeg,decDeg,separationArcsec,physical_separation_kpc,z,photoZ,catalogue_table_name,z_is_spec,z_best
0,170019695496790075,53.024724,-28.329677,12,61088.093945,61358.267723,1352.51,SN,53.012670,-28.358580,110.866,10.361,0.004526,NaN,LASR,True,0.004526
1,170019696304193611,59.426619,-48.908000,15,61088.098045,61095.060653,1444.64,SN,59.426780,-48.908470,1.926,0.150,0.003766,NaN,DESI/LASR,True,0.003766
2,170019696393848492,65.265153,-48.303625,21,61088.098476,61095.085069,1152.02,SN,65.266052,-48.303947,1.621,0.516,0.015614,NaN,LASR,True,0.015614
3,170019716255973493,149.337745,0.960426,902,61088.211979,61160.072531,3043.03,SN,149.337952,0.960681,1.185,NaN,NaN,0.192099,SDSS/PS1/DESI,False,0.192099
4,170019716260167879,149.165981,1.235018,11,61088.211979,61095.161163,-1203.98,NT,149.166042,1.235139,0.755,1.507,0.109487,NaN,NED/SDSS,True,0.109487


### Every host-z transient in Lasair, up to N

Rather than checking the ALeRCE ids one batch at a time, query the whole objects table with the same
cut. Each page returns up to 10,000 rows, so the full set (83,677 `SN`/`NT` with a host redshift as of
2026-11-20; 11,075 of those with `nDiaSources > 10`) costs about 9 calls. Running all 100k ALeRCE ids
through `lasair_for_ids` would cost about 486 calls. `max_rows` caps the pull. Paging is keyset on
`diaObjectId`, as in `recent_objects`.

`in_alerce` flags which rows are also in the ALeRCE id list loaded above.

In [15]:
def host_z_transients(max_rows=100_000, extra_conditions=None, cuts=HOST_Z_TRANSIENTS,
                      page_size=MAX_PAGE, verbose=True):
    """Every Lasair object passing `cuts` (default: SN/NT with a host z), up to `max_rows`.

    extra_conditions: more SQL ANDed on, e.g. "objects.nDiaSources > 10".
    Costs ceil(min(max_rows, total) / page_size) calls; count it first with
    L.query(selected="count(*) AS n", tables="objects, sherlock_classifications", ...).
    """
    selected, tables = MODES["hosts"]
    conditions = cuts + (f" AND ({extra_conditions})" if extra_conditions else "")

    rows, last_id = [], None
    while len(rows) < max_rows:
        page_conditions = conditions
        if last_id is not None:
            page_conditions += f" AND objects.diaObjectId > {last_id}"
        page_conditions += " ORDER BY objects.diaObjectId"
        limit = min(page_size, MAX_PAGE, max_rows - len(rows))

        page = L.query(selected=selected, tables=tables, conditions=page_conditions, limit=limit)
        if isinstance(page, dict) and "error" in page:
            raise RuntimeError(page["error"])
        rows.extend(page)
        if verbose:
            print(f"call: +{len(page)} rows (total {len(rows)})")
        if len(page) < limit:
            break                                  # table exhausted
        last_id = page[-1]["diaObjectId"]

    df = pd.DataFrame(rows)
    if not df.empty:
        df["z_is_spec"] = df.z.notna()
        df["z_best"] = df.z.fillna(df.photoZ)
    return df


N_MAX = 100_000   # above the current total, so this pulls everything (~9 calls)

all_hosted = host_z_transients(max_rows=N_MAX)
all_hosted["in_alerce"] = all_hosted.diaObjectId.isin(oids)
all_hosted.to_csv("lasair_hostz_transients.csv", index=False)

print(f"\n{len(all_hosted)} transients with a host redshift")
print(all_hosted.classification.value_counts().to_string())
print(f"spec-z: {all_hosted.z_is_spec.sum()}, photo-z only: {(~all_hosted.z_is_spec).sum()}")
print(f"also in the ALeRCE list: {all_hosted.in_alerce.sum()}")
all_hosted.head()

call: +10000 rows (total 10000)
call: +10000 rows (total 20000)
call: +10000 rows (total 30000)
call: +10000 rows (total 40000)
call: +10000 rows (total 50000)
call: +10000 rows (total 60000)
call: +10000 rows (total 70000)
call: +10000 rows (total 80000)
call: +3677 rows (total 83677)

83677 transients with a host redshift
SN    74017
NT     9660
spec-z: 40763, photo-z only: 42914
also in the ALeRCE list: 7481


,diaObjectId,ra,decl,nDiaSources,firstDiaSourceMjdTai,lastDiaSourceMjdTai,latest_psfFlux,classification,raDeg,decDeg,separationArcsec,physical_separation_kpc,z,photoZ,catalogue_table_name,z_is_spec,z_best,in_alerce
0,170019695496790075,53.024724,-28.329677,12,61088.093945,61358.267723,1352.51,SN,53.012670,-28.358580,110.866,10.361,0.004526,NaN,LASR,True,0.004526,True
1,170019695992242795,52.823847,-29.691600,7,61088.093945,61107.030002,7855.31,SN,52.823875,-29.692472,3.138,6.290,0.109912,NaN,NED,True,0.109912,False
2,170019696062496843,51.729318,-27.760201,4,61088.093945,61099.042416,1890.92,SN,51.730043,-27.760486,2.529,20.446,1.044000,NaN,MILLIQUAS,True,1.044000,False
3,170019696211394730,51.952526,-26.746976,2,61088.097446,61357.067176,2292.75,NT,51.952500,-26.747060,0.313,0.940,0.177900,NaN,NED,True,0.177900,False
4,170019696300523640,60.151232,-49.046520,192,61088.098045,61105.093204,3276.23,SN,60.122500,-49.029860,90.462,6.606,0.003532,NaN,LASR,True,0.003532,False


In [17]:
all_hosted.to_csv("lasair_for_alerce_haveHostsWZ_lim100k.csv")